# Ray Variables
computing all variables used in computing $D$, $\Phi^2$, $\Lambda$, and $F_{\phi, \chi}$ along the each ray.
And debug them for errors

## Multiple Ray Paths

In [ ]:
from nescafe23_fluctuations import tools
import numpy as np
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv, find_dotenv
import pickle

load_dotenv(find_dotenv())

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

In [ ]:
fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/ray_run_forward_back.pkl"


with open(fn, 'rb') as f:
    ray_run = pickle.load(f)

rays = ray_run['rays']
eigs = ray_run['eigs']

rays_back = ray_run['rays_back']
eigs_back = ray_run['eigs_back']

source_depth = ray_run["geo"]["source_depth"]
reciever_range = ray_run["geo"]["receiver_range"]
num_range_save = 1000

envi = ray_run["envi"]

In [ ]:
latitude = 39.192499999
Omega = 7.292115e-5  # Earth Rotation Speed [rad/s]
f = 2 * Omega * np.sin(np.radians(latitude))  # [rad/s]

zeta0 = 7.3 * np.sqrt(2)
rvs = tools.compute_n23_ray_variables(f=f, zeta0=zeta0)

In [ ]:
tools.plot_ray_variables(rvs, ['-8','+7'])

In [ ]:
tools.plot_ray_variables(rvs, ['+6'])

In [ ]:
tools.plot_ray_variables(rvs, ['-6'], pair_idxs={'-6':[0,1,2]})

In [ ]:
Ds = {}

dzr = np.arange(0,500, 2)

for ray_id in ['+7','-7','+6','-6','-5b']:
    Ds[ray_id] = []
    for rv in rvs[ray_id]:
        Ds[ray_id].append(tools.compute_phase_structure_function(rv, dzr))

In [ ]:
plt.scatter(rays.ts[:,-1], rays.zs[:,-1], lw=0, s=5, c='k')

plt.plot(rays.ts[:,-1], rays.zs[:,-1], lw=0.5, c='k')
plt.xlim([100.2,101.5])

plt.xlim([101.1, 101.2])
plt.ylim([-1250,250])
plt.xlabel('time [s]')
plt.ylabel('depth [m]')

In [ ]:

xlim = (100,102)
for ray_id in tools.EIG_INDEXS.keys():
    ray_id_pr,_ = tools.parse_ray_id(ray_id)
    for pair_idx in range(len(tools.EIG_INDEXS[ray_id])):
        eig_indx = np.where(eigs.ray_id[0] == ray_id_pr)[0][tools.EIG_INDEXS[ray_id][pair_idx][0]]
        plt.scatter(eigs.ts[0][eig_indx,-1]-0.235, eigs.received_angles[0][eig_indx], facecolors='none', lw=0.8, s=35, edgecolors=(0, 0, 0, 0.9))
    
    if eigs.ts[0][eig_indx,-1]-0.235 > xlim[1]:
        continue
    plt.text(eigs.ts[0][eig_indx,-1]-0.235 + 0.05, eigs.received_angles[0][eig_indx] + 1, ray_id, fontsize=7)

plt.xlim(xlim)
plt.xlabel('time [s]')
plt.ylabel('receive angle [deg]')

In [ ]:
for ray_id in ['+7']:

    for rv in rvs[ray_id]:
        plt.plot(rv['rs']/1000, rv['zs'])

plt.xlabel('range [km]')
plt.ylabel('depth [m]')
plt.ylim([-5000,0])

In [ ]:
for ray_id in ['-7']:

    for rv in rvs[ray_id]:
        plt.plot(rv['rs']/1000, rv['zs'])

plt.xlabel('range [km]')
plt.ylabel('depth [m]')
plt.ylim([-5000,0])

In [ ]:
for ray_id in ['-5b']:

    for rv in rvs[ray_id]:
        plt.plot(rv['rs']/1000, rv['zs'])

plt.xlabel('range [km]')
plt.ylabel('depth [m]')
plt.ylim([-5000,0])

In [ ]:
plt.plot(rvs['-7'][0]['RT_zeta_1'])
plt.plot(rvs['-7'][1]['RT_zeta_1'])
plt.plot(rvs['-7'][1]['RT_zeta_1'])

In [ ]:
fig,axes=plt.subplots(2,3, figsize=(13,6))
axes=axes.flatten()

for k, ray_id in enumerate(['+7','-7','+6', '-6', '-5b']):
    plt.sca(axes[k])
    for D in Ds[ray_id]:
        plt.plot(dzr, np.exp(-D/2))
    plt.title(ray_id)
    plt.xlabel('$\\Delta z_r$')
    plt.grid()


plt.tight_layout()

In [ ]:
plt.close()
plt.plot(np.arange(10))

In [ ]:
plt.close()
fig,axes=plt.subplots(2,3, figsize=(13,6))
axes=axes.flatten()

for k, ray_id in enumerate(['+7','-7','+6', '-6', '-5b']):
    plt.sca(axes[k])
    for D in Ds[ray_id]:
        plt.plot(dzr, np.exp(-D/2))
    plt.title(ray_id)
    plt.xlabel('$\\Delta z_r$')
    plt.grid()


plt.tight_layout()

In [ ]:
tools.plot_ray_variables(rvs, ['+7','-7', '+6', '-6'], pair_idxs = {'+7':[0], '-7':[0], '+6':[0], '-6':[0]})

## Number of IW modes along ray path

In [ ]:
for D in Ds['-6']:
    plt.plot(dzr, np.exp(-D/2))
plt.title('-6')

In [ ]:
plt.imshow(np.log(rvs['-7']['H']))
plt.colorbar()

In [ ]:
plt.plot(rvs['-7']['rs']/1000, rvs['-7']['J'])
plt.xlabel('range [km]')
plt.ylabel('number of IW modes')

In [ ]:
#plt.plot(rvs['-7']['mean_j'])
plt.plot(rvs['-7']['mean_jn1'])

In [ ]:
tools.plot_ray_variables(rvs, ['+6', '-6'], 0)

In [ ]:
summands = {}

for ray_id in ['-7', '+7', '-6', '+6']:
    
    rv = rvs[ray_id]
    fc = 250
    deltaz_r = np.arange(0,500, 2)
    
    omegac = 2 * np.pi * fc
    
    ## Compute phase structure function at deltaz_r locations
    mj = (
        np.pi
        * rv["j"][None, :, None]
        * rv["Nray"].values[:, None, None]
        / rv["N0B_ray"].values[:, None, None]
    )  # (s,j,1)
    dz_s = rv["RT_zeta_1"][:, None, None] * deltaz_r[None, None, :]  # (s,1,dzr)
    summand = rv["H"].T[:, :, None] * (
        (1 - np.cos(mj * dz_s)) / rv["j"][None, :, None]
    )  # (s,j,dzr)
    # set any nan values to zero
    summand = np.where(np.isfinite(summand), summand, 0)

    summands[ray_id] = summand

In [ ]:
for ray_id in ['-7', '+7', '-6', '+6']:
    plt.plot(

In [ ]:
rvs[ray_id].keys()

In [ ]:
(rvs[ray_id]['Nray'] / rvs[ray_id]['N0B_ray']).plot()

In [ ]:
for k, ray_id in enumerate(['-7', '+7', '-6', '+6']):
    rv = rvs[ray_id]

    ## Compute phase structure function at deltaz_r locations
    mj = (
        np.pi
        * rv["j"][None, :, None]
        * rv["Nray"].values[:, None, None]
        / rv["N0B_ray"].values[:, None, None]
    )  # (s,j,1)


    dz_s = rv["RT_zeta_1"][:, None, None] * deltaz_r[None, None, :]  # (s,j,dzr)
    cos_term = (mj * dz_s)[:,:,50]

    plt.plot(np.log(np.abs(cos_term)), c=f'C{k}', alpha=0.01)

plt.hlines(np.log(2*np.pi), 0, 1000, colors='k') 

In [ ]:
for k, ray_id in enumerate(['-7', '+7', '-6', '+6']):
    _ = plt.plot(rvs[ray_id]['rs']/1000, summands[ray_id][:,:,50], alpha=0.1, c=f'C{k}')

In [ ]:
for ray_id in ['-7', '+7', '-6', '+6']:
    plt.plot(rvs[ray_id]['rs']/1000, np.sum(summands[ray_id][:,:,75], axis=1))

In [ ]:
_ = plt.plot(rv['rs']/1000, summand[:,:,75], lw=1, c='k', alpha=0.05)
plt.yscale('log')

In [ ]:
for ray_id in ['-7', '+7', '-6', '+6']:
    plt.plot(rvs[ray_id]['rs']/1000, rvs[ray_id]['zs'])
    plt.fill_between(rvs[ray_id]['rs']/1000, rvs[ray_id]['zs'] + np.sqrt(rvs[ray_id]['Rf2'])/2, rvs[ray_id]['zs'] - np.sqrt(rvs[ray_id]['Rf2'])/2, alpha=0.3)
    

In [ ]:
D = {}
Lamda = {}
Phi2 = {}
F_phi = {}
F_chi = {}

dzr = np.arange(0,500, 2)
m = 2 * np.pi * np.logspace(-5, 0, 500)[None, :]  # [rad / m]

for ray_id in ['-6', '+6', '-7', '+7']:
    D[ray_id] = tools.compute_phase_structure_function(rvs[ray_id], dzr)
    Lamda[ray_id], Phi2[ray_id] = res = tools.compute_lambda_phi(rvs[ray_id])
    F_phi[ray_id], F_chi[ray_id] = tools.compute_F_phi_chi(rvs[ray_id], m)

In [ ]:
plt.scatter(list(Lamda.values()), np.sqrt(list(Phi2.values())))
plt.yscale('log')
plt.xscale('log')

plt.ylim([1e0, 1e2])
plt.xlim([1e-2, 1e0])

plt.grid()

In [ ]:
for ray_id in ['-7', '+7', '-6', '+6']:
    plt.plot(dzr, np.exp(-D[ray_id]/2), label=ray_id)

plt.legend()

In [ ]:
fig = plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
for ray_id in D:
    plt.plot(m[0]/(2*np.pi), F_phi[ray_id], label=ray_id)

plt.legend()


plt.yscale('log')
plt.xscale('log')

plt.subplot(1,2,2)
for ray_id in D:
    plt.plot(m[0]/(2*np.pi), F_chi[ray_id], label=ray_id)

plt.legend()


plt.yscale('log')
plt.xscale('log')

### investigate ray seperation

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
import pickle
import matplotlib.pyplot as plt

_ = load_dotenv(find_dotenv())

In [ ]:
fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/ray_run_forward_back.pkl"

with open(fn, 'rb') as f:
    ray_run = pickle.load(f)

rays = ray_run['rays']
eigs = ray_run['eigs']

In [ ]:
ray_seperation = rays.zs[1:,:] - rays.zs[:-1,:]

In [ ]:
%matplotlib widget

In [ ]:
plt.close()
plt.imshow(np.log(np.abs(ray_seperation)), aspect='auto', extent = (rays.rs[0,0], rays.rs[0,-1], rays.thetas[0], rays.thetas[-1]))
plt.colorbar()

plt.hlines(eigs.received_angles[0], 0, 150000, lw=1, colors='k', alpha=0.3)

In [ ]:
eig_idx = np.where(eigs.ray_id[0] == '7.0')[0]

In [ ]:
eig_idx

In [ ]:
plt.close()
plt.plot(eigs.rs[0][eig_idx,:].T, eigs.zs[0][eig_idx,:].T)

In [ ]:
eigs.received_angles

## Investigate Vertical Coherence

In [ ]:
from nescafe23_fluctuations import tools
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv, find_dotenv
import os
import pickle

load_dotenv(find_dotenv())

%load_ext autoreload
%autoreload 2

In [ ]:
latitude = 39.192499999
Omega = 7.292115e-5  # Earth Rotation Speed [rad/s]
f = 2 * Omega * np.sin(np.radians(latitude))  # [rad/s]

zeta0 = 7.3 * np.sqrt(2)
rvs = tools.compute_n23_ray_variables(f=f, zeta0=zeta0)

In [ ]:
plt.plot(rvs['+6'][0]['wronk'])

In [ ]:
plt.plot(rvs['+7'][0]['rs'], rvs['+7'][0]['zs'])
plt.plot(rvs['+7'][1]['rs'], rvs['+7'][1]['zs'])

In [ ]:
tools.plot_ray_variables(rvs, ['+7', '-7'], pair_idxs=[0])

In [ ]:
tools.plot_ray_variables(rvs, ['+7', '-7', '-6', '+6'], pair_idxs=[0])

In [ ]:
fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/ray_run_forward_back.pkl"

with open(fn, 'rb') as f:
    ray_run = pickle.load(f)

In [ ]:
eig_idx_forward = np.where(ray_run['eigs'].ray_id[0] == '6.0')[0][0]
eig_idx_backward = np.where(ray_run['eigs_back'].ray_id[0] == '6.0')[0][0]

In [ ]:
plt.plot(rvs['+6'][0]['rs'], rvs['+6'][0]['zs'])
plt.plot(rvs['+6'][0]['rs_back'], np.flip(rvs['+6'][0]['zs_back']))

In [ ]:
plt.subplot(1,2,1)
plt.plot(rvs['+6'][0]['rs']/1000, (rvs['+6'][0]['RT_zeta_1'] * dx2) - (rvs['+6'][0]['RT_zeta_2'] * dx1))

ax2 = plt.twinx()
plt.plot(rvs['+6'][0]['rs']/1000, (rvs['+6'][0]['RT_zeta_1'] * rvs['+6'][0]['RT_zeta_2'] ), c='C1')

plt.subplot(1,2,2)
plt.plot(rvs['+6'][0]['rs']/1000, (rvs['+6'][0]['Rf2']))

plt.tight_layout()

In [ ]:
ray_id = '-7'

plt.plot(rvs[ray_id][0]['RT_zeta_1'])
plt.plot(rvs[ray_id][0]['RT_zeta_2'])

plt.plot(rvs[ray_id][0]['RT_zeta_1_2'])
plt.plot(rvs[ray_id][0]['RT_zeta_2_2'])

In [ ]:
plt.close()

plt.plot(ray_run['envi'].sound_speed.sel({'range':30000}, method='nearest').differentiate('depth'), ray_run['envi'].sound_speed.depth)


plt.hlines(250, -1,1)

plt.ylim([1000,0])

In [ ]:
rvs['+6'][0]['zs'].max()

In [ ]:
ray_run['envi'].sound_speed.differentiate('depth').plot()


plt.plot(rvs['+6'][0]['rs'], -rvs['+6'][0]['zs'])
#plt.plot(rvs['-6'][0]['rs'], -rvs['-6'][0]['zs'])

plt.ylim([400,200])

## Exploring fresnel zone estimation methods

In [ ]:
import pygenray as pr

In [ ]:
ray_id = '+7'

fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/ray_run_forward_back.pkl"


with open(fn, 'rb') as f:
    ray_run = pickle.load(f)

rays = ray_run['rays']
eigs = ray_run['eigs']

rays_back = ray_run['rays_back']
eigs_back = ray_run['eigs_back']

source_depth = ray_run["geo"]["source_depth"]
reciever_range = ray_run["geo"]["receiver_range"]
num_range_save = 1000

envi = ray_run["envi"]

In [ ]:
eig_idx_forward = 0

# Convert backward ray_id to pr format
ray_id_pr, _ = tools.parse_ray_id(ray_id)

try:
    ray_idx_forward = np.where(eigs.ray_id[0] == ray_id_pr)[0][eig_idx_forward]
except IndexError:
    ray_idx_forward = np.where(eigs.ray_id[0] == ray_id_pr)[0]



forward_launch_angle = eigs.launch_angles[0][ray_idx_forward]
#backward_launch_angle = eigs_backward.launch_angles[0][ray_idx_backward]

forward_launch_angles_hat = np.linspace(forward_launch_angle-10, forward_launch_angle + 10, 2000)
#backward_launch_angles_hat = np.linspace(backward_launch_angle - np.degrees(1), backward_launch_angle + np.degrees(1), 400)

In [ ]:
forward_launch_angle

In [ ]:
ray_hat_forward = pr.shoot_rays(
    source_depth, 0, forward_launch_angles_hat, reciever_range, num_range_save, envi
)

In [ ]:
delta_t = eigs.ts[0][ray_idx_forward][:,None] - ray_hat_forward.ts.T
delta_phi = 2 * np.pi * delta_t * 250

In [ ]:
fig = plt.figure(figsize=(10,4))
_ = plt.plot(ray_hat_forward.rs.T, delta_phi/np.pi, c='k', alpha=0.5)
plt.hlines([1, -1], 0, ray_hat_forward.rs[0,-1])

plt.ylim([-10,10])

In [ ]:
delta_phi[150]

In [ ]:
ray_hat_forward.rs[0,150]

In [ ]:
fig = plt.figure(figsize=(10,4))

plt.imshow(np.abs(delta_phi - np.pi).T, aspect='auto')

plt.colorbar()

In [ ]:
def find_first_zero_crossing(arr, axis=0):
    """
    Find the first zero crossing along a specified axis.
    
    Parameters:
    -----------
    arr : np.ndarray
        Input array of any shape
    axis : int
        Axis along which to find zero crossings
    
    Returns:
    --------
    np.ndarray or None
        Array of indices where first zero crossing occurs along the axis.
        Shape will be arr.shape with the specified axis removed.
    """
    signs = np.sign(arr)
    
    # Compare consecutive elements along the specified axis
    sign_changes = np.take(signs, range(signs.shape[axis] - 1), axis=axis) * \
                   np.take(signs, range(1, signs.shape[axis]), axis=axis) < 0
    
    # Find first occurrence along axis
    # Use argmax which returns first True index (or 0 if none)
    first_crossing = np.argmax(sign_changes, axis=axis)
    
    # Mask where no crossing was found
    has_crossing = np.any(sign_changes, axis=axis)
    first_crossing = np.where(has_crossing, first_crossing, np.nan)
    
    return first_crossing

In [ ]:
pi_crossings = find_first_zero_crossing(delta_phi - np.pi, axis=1)
mpi_crossings = find_first_zero_crossing(delta_phi + np.pi, axis=1)

In [ ]:
fig = plt.figure(figsize=(10,4))

plt.imshow(delta_phi.T, aspect='auto', vmax=100, vmin=-100, cmap='RdBu')

plt.plot(find_first_zero_crossing(delta_phi - np.pi/2, axis=1))
plt.plot(find_first_zero_crossing(delta_phi + np.pi/2, axis=1))
plt.colorbar()

plt.vlines(35,2000,0)

In [ ]:
z1 = []
z2 = []

for k in range(1000):
    if np.isnan(pi_crossings[k]):
        z1.append(np.nan)
    else:
        z1.append(ray_hat_forward.zs[int(pi_crossings[k]), k])

    if np.isnan(mpi_crossings[k]):
        z2.append(np.nan)
    else:
        z2.append(ray_hat_forward.zs[int(mpi_crossings[k]), k])

z1 = np.array(z1)
z2 = np.array(z2)

In [ ]:
plt.plot(np.sqrt(rvs['+7'][0]['Rf2'])[60:])
plt.plot(np.abs(z2-z1)[60:])